In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"

df = pd.read_csv(url)

print(df.head())
print(df.shape)

  Customer          ST GENDER             Education Customer Lifetime Value  \
0  RB50392  Washington    NaN                Master                     NaN   
1  QZ44356     Arizona      F              Bachelor              697953.59%   
2  AI49188      Nevada      F              Bachelor             1288743.17%   
3  WW63253  California      M              Bachelor              764586.18%   
4  GA49547  Washington      M  High School or Below              536307.65%   

    Income  Monthly Premium Auto Number of Open Complaints     Policy Type  \
0      0.0                1000.0                    1/0/00   Personal Auto   
1      0.0                  94.0                    1/0/00   Personal Auto   
2  48767.0                 108.0                    1/0/00   Personal Auto   
3      0.0                 106.0                    1/0/00  Corporate Auto   
4  36357.0                  68.0                    1/0/00   Personal Auto   

   Vehicle Class  Total Claim Amount  
0  Four-Door Car 

In [2]:
# Exercise 1: Cleaning column names

df.columns = df.columns.str.lower().str.replace(" ", "_")

df = df.rename(columns={
    "st": "state"
})

print(df.columns.tolist())

['customer', 'state', 'gender', 'education', 'customer_lifetime_value', 'income', 'monthly_premium_auto', 'number_of_open_complaints', 'policy_type', 'vehicle_class', 'total_claim_amount']


In [3]:
# Exercise 2: Cleaning invalid values

# Clean gender
df["gender"] = df["gender"].replace({
    "Female": "F",
    "female": "F",
    "Femal": "F",
    "Male": "M",
    "male": "M"
})

# Clean state
df["state"] = df["state"].replace({
    "AZ": "Arizona",
    "Cali": "California",
    "WA": "Washington"
})

# Clean education
df["education"] = df["education"].replace({
    "Bachelors": "Bachelor"
})

# Clean customer lifetime value
df["customer_lifetime_value"] = (
    df["customer_lifetime_value"]
    .str.replace("%", "", regex=False)
)

# Clean vehicle class
df["vehicle_class"] = df["vehicle_class"].replace({
    "Sports Car": "Luxury",
    "Luxury SUV": "Luxury",
    "Luxury Car": "Luxury"
})

print(df.head())

  customer       state gender             education customer_lifetime_value  \
0  RB50392  Washington    NaN                Master                     NaN   
1  QZ44356     Arizona      F              Bachelor               697953.59   
2  AI49188      Nevada      F              Bachelor              1288743.17   
3  WW63253  California      M              Bachelor               764586.18   
4  GA49547  Washington      M  High School or Below               536307.65   

    income  monthly_premium_auto number_of_open_complaints     policy_type  \
0      0.0                1000.0                    1/0/00   Personal Auto   
1      0.0                  94.0                    1/0/00   Personal Auto   
2  48767.0                 108.0                    1/0/00   Personal Auto   
3      0.0                 106.0                    1/0/00  Corporate Auto   
4  36357.0                  68.0                    1/0/00   Personal Auto   

   vehicle_class  total_claim_amount  
0  Four-Door Car 

In [4]:
# Exercise 3: Formatting data types

# Convert customer lifetime value to numeric
df["customer_lifetime_value"] = pd.to_numeric(
    df["customer_lifetime_value"],
    errors="coerce"
)

# Check unique complaint values
print(df["number_of_open_complaints"].unique())

# Extract the middle number
df["number_of_open_complaints"] = (
    df["number_of_open_complaints"]
    .astype("string")
    .str.split("/")
    .str[1]
)

# Convert complaints to numeric
df["number_of_open_complaints"] = pd.to_numeric(
    df["number_of_open_complaints"],
    errors="coerce"
)

print(df.dtypes)

<ArrowStringArray>
['1/0/00', '1/2/00', '1/1/00', '1/3/00', '1/5/00', '1/4/00', nan]
Length: 7, dtype: str
customer                         str
state                            str
gender                           str
education                        str
customer_lifetime_value      float64
income                       float64
monthly_premium_auto         float64
number_of_open_complaints    float64
policy_type                      str
vehicle_class                    str
total_claim_amount           float64
dtype: object


In [5]:
# Exercise 4: Handling missing values

print("Missing values before cleaning:")
print(df.isnull().sum())

# Fill categorical missing values with mode
categorical_columns = df.select_dtypes(
    include=["object", "string", "category"]
).columns

for col in categorical_columns:
    if df[col].notna().any():
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna("Unknown")

# Fill numerical missing values with median
numerical_columns = df.select_dtypes(
    include="number"
).columns

for col in numerical_columns:
    df[col] = df[col].fillna(df[col].median())

# Convert numerical columns to integers
df[numerical_columns] = df[numerical_columns].astype(int)

print("\nMissing values after cleaning:")
print(df.isnull().sum())

print("\nUpdated data types:")
print(df.dtypes)

Missing values before cleaning:
customer                     2937
state                        2937
gender                       3054
education                    2937
customer_lifetime_value      2940
income                       2937
monthly_premium_auto         2937
number_of_open_complaints    2937
policy_type                  2937
vehicle_class                2937
total_claim_amount           2937
dtype: int64

Missing values after cleaning:
customer                     0
state                        0
gender                       0
education                    0
customer_lifetime_value      0
income                       0
monthly_premium_auto         0
number_of_open_complaints    0
policy_type                  0
vehicle_class                0
total_claim_amount           0
dtype: int64

Updated data types:
customer                       str
state                          str
gender                         str
education                      str
customer_lifetime_value      int64

In [6]:
# Exercise 5: Handling duplicates

print("Duplicates before cleaning:")
print(df.duplicated().sum())

# Remove duplicate rows
df = df.drop_duplicates()

# Reset index
df = df.reset_index(drop=True)

print("Duplicates after cleaning:")
print(df.duplicated().sum())

# Save cleaned dataset
df.to_csv("cleaned_insurance_data.csv", index=False)

print("Cleaned dataset saved successfully!")
print("Final dataset shape:", df.shape)

Duplicates before cleaning:
2936
Duplicates after cleaning:
0
Cleaned dataset saved successfully!
Final dataset shape: (1072, 11)


In [7]:
# Bonus Challenge 3

print("Summary statistics:")
print(df[[
    "total_claim_amount",
    "customer_lifetime_value"
]].describe())

# Calculate percentiles
high_claim_threshold = df["total_claim_amount"].quantile(0.75)
low_clv_threshold = df["customer_lifetime_value"].quantile(0.25)

# Filter customers
high_claim_low_clv = df.loc[
    (df["total_claim_amount"] > high_claim_threshold) &
    (df["customer_lifetime_value"] <= low_clv_threshold)
]

print("\nNumber of selected customers:")
print(len(high_claim_low_clv))

print("\nSummary statistics:")
print(
    high_claim_low_clv[
        ["total_claim_amount", "customer_lifetime_value"]
    ].describe()
)

print("\nFirst 5 customers:")
print(high_claim_low_clv.head())

Summary statistics:
       total_claim_amount  customer_lifetime_value
count         1072.000000             1.072000e+03
mean           404.481343             7.929229e+05
std            292.909076             6.423980e+05
min              0.000000             2.004350e+05
25%            201.750000             4.054045e+05
50%            354.000000             5.881740e+05
75%            532.000000             8.937668e+05
max           2893.000000             5.816655e+06

Number of selected customers:
40

Summary statistics:
       total_claim_amount  customer_lifetime_value
count           40.000000                40.000000
mean           723.300000            341577.425000
std            149.216449             56921.466178
min            537.000000            228759.000000
25%            632.750000            304220.500000
50%            685.000000            365995.000000
75%            799.000000            387364.000000
max           1185.000000            402381.000000

First 

In [8]:
# Remove rows without an original customer ID
df = df[df["customer"].notna() & (df["customer"] != "nan")]

# Remove duplicates and reset index
df = df.drop_duplicates().reset_index(drop=True)

# Save the cleaned dataset
df.to_csv("cleaned_insurance_data.csv", index=False)

print("Final dataset shape:", df.shape)
print("Duplicates:", df.duplicated().sum())

Final dataset shape: (1072, 11)
Duplicates: 0


In [10]:
# Remove artificial rows without valid customer IDs
df = df[df["customer"].str.fullmatch(r"[A-Za-z]{2}\d{5}", na=False)]

# Reset index
df = df.reset_index(drop=True)

# Save final dataset
df.to_csv("cleaned_insurance_data.csv", index=False)

print("Final dataset shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())

Final dataset shape: (1072, 11)
Missing values: 0
Duplicates: 0


In [11]:
# Bonus Challenge 3: Analyzing cleaned data

# Calculate percentiles
high_claim = df["total_claim_amount"].quantile(0.75)
low_clv = df["customer_lifetime_value"].quantile(0.25)

# Filter customers
selected_customers = df[
    (df["total_claim_amount"] > high_claim) &
    (df["customer_lifetime_value"] <= low_clv)
]

# Display results
print("Number of selected customers:", len(selected_customers))

print("\nSummary statistics:")
print(selected_customers[
    ["total_claim_amount", "customer_lifetime_value"]
].describe())

print("\nFirst 5 customers:")
print(selected_customers.head())

Number of selected customers: 40

Summary statistics:
       total_claim_amount  customer_lifetime_value
count           40.000000                40.000000
mean           723.300000            341577.425000
std            149.216449             56921.466178
min            537.000000            228759.000000
25%            632.750000            304220.500000
50%            685.000000            365995.000000
75%            799.000000            387364.000000
max           1185.000000            402381.000000

First 5 customers:
   customer       state gender education  customer_lifetime_value  income  \
17  OE15005  California      F   College                   394524   28855   
23  TZ98966      Nevada      F  Bachelor                   245019       0   
26  US89481  California      F  Bachelor                   394637       0   
38  CM61827      Oregon      F  Bachelor                   309003       0   
42  IL19217  California      F  Bachelor                   393900   60021   

    